# Testing differences and communicating findings

Many analytical questions compare groups: do entire homes cost more than private rooms, is a night in Mitte more expensive than in Neukölln, do hosts with several listings fill in their registration details more often, does a new listing page lead to more bookings? This page recaps the tests for such questions as tools to **choose, run and interpret**: the t-test and the Mann–Whitney test for numeric outcomes, the chi-square test and Cramér's V for categorical outcomes, and the A/B test as the main application. Every test result is reported with a **confidence interval** and an **effect size**, because with thousands of listings almost any difference becomes "significant". The theory behind the tests belongs to the statistics module; workbook [07](07-hypothesis-testing.ipynb) and the extras notebook [11](extras/11-hypothesis-testing-by-simulation.ipynb) recap it. The page ends with the step from correlation to the regression line, the bridge to Session 6, and with how to communicate findings to a non-technical reader: a one-page report or a small Streamlit dashboard.

```mermaid
flowchart LR
    A["question:<br/>do groups differ?"] --> B["choose a test<br/>(decision table)"]
    B --> C["run it:<br/>statistic, p-value"]
    C --> D["estimate:<br/>difference with 95 % CI"]
    D --> E["effect size:<br/>d, V, CLES"]
    E --> F["interpret in units<br/>the reader understands"]
```

In [1]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

## Comparing groups: the logic of a test

### Concept

Every test follows the same recipe (Downey: "There is only one test"):

1. Choose a **test statistic** that measures the effect, such as the difference in means.
2. State the **null hypothesis H₀**: a world in which the effect does not exist.
3. Work out how the statistic varies in that world, by formula or by simulation.
4. The **p-value** is the probability, in that world, of a statistic at least as extreme as the observed one.

A p-value is **not** the probability that H₀ is true, and not a measure of how large the effect is. The threshold 0.05 (the **significance level α**) is a convention.

A **confidence interval (CI)** gives a range of plausible values for the effect. A 95 % CI is produced by a procedure that captures the true value in 95 % of repeated studies. If the 95 % CI for a difference excludes 0, the two-sided test at α = 0.05 rejects H₀; the CI tells you in addition how large the difference could be.

### Why it matters

The p-value answers only "could this be zero?". A stakeholder needs "how much, in units I understand, and how sure are we?". The American Statistical Association's statement on p-values (Wasserstein & Lazar, 2016) asks for estimates with intervals instead of a bare "significant / not significant".

### How it works in Python

A **permutation test** builds the null world directly: if the district did not matter for the price, shuffling the district labels between Mitte and Neukölln listings would not change the difference in mean price.

In [2]:
import pandas as pd
from scipy import stats

listings = pd.read_parquet("case-study/data/airbnb/listings.parquet")
short = listings[listings["price"].notna() & listings["minimum_nights"].lt(28)]   # comparable prices (Session 4)
mitte = short.loc[short["district"] == "Mitte", "price"].to_numpy()
neukoelln = short.loc[short["district"] == "Neukölln", "price"].to_numpy()
print(len(mitte), len(neukoelln), round(mitte.mean(), 1), round(neukoelln.mean(), 1))   # 1537 556 220.4 154.9

res = stats.permutation_test((mitte, neukoelln), lambda a, b: a.mean() - b.mean(),
                             n_resamples=2_000, random_state=0)
print(round(res.statistic, 1), res.pvalue)         # 65.5 0.0009995: no shuffle came close

1537 556 220.4 154.9
65.5 0.0009995002498750624


With 2,000 shuffles the smallest possible p-value is 1/2,001 ≈ 0.0005; "p < 0.001" is the honest report.

> **Caution.** **Significant is not important.** With large samples, tiny differences reach p < 0.05. With small samples, large differences may not. Always report the size of the effect next to the p-value.

## t-test and Mann–Whitney test with confidence intervals and effect sizes

### Concept

- **Welch's t-test** compares two **means** without assuming equal variances. Use it by default; the equal-variance "Student" version is rarely needed. It relies on the sample means being approximately normal, which holds for large samples even when the data are skewed (central limit theorem).
- The **Mann–Whitney U test** compares **ranks**: does a random value from one group tend to exceed a random value from the other? It is robust to outliers and suits skewed data such as prices and ordinal data such as star ratings.
- If the same units are measured twice (before and after), use the **paired** versions: the paired t-test or the Wilcoxon signed-rank test.

Effect sizes:

- the **difference in means** with its CI, in real units (euros, nights, stars): the most useful for readers;
- **Cohen's d** = difference in means / pooled SD. Rough guide: 0.2 small, 0.5 medium, 0.8 large;
- the **common-language effect size (CLES)** = U / (n₁ · n₂): the probability that a random observation from group 1 exceeds one from group 2 (ties count half). 0.5 means no effect.

Worked example: short-stay entire homes cost €226 on average, private rooms €116, pooled SD about €212. Then d ≈ (226 − 116) / 212 ≈ 0.52: a "medium" difference by the rough guide. Yet a random entire home costs more than a random private room in 84 % of pairs (CLES 0.84). The two effect sizes disagree because a few prices in the thousands inflate the SD; on the log scale, d is 1.26. For skewed data, prefer the CLES or compute d on a sensible scale.

### Why it matters

Two tests answer slightly different questions (means versus "tends to be larger"). Running both is a robustness check: when they agree, the conclusion does not depend on the choice. Effect sizes let you compare results across datasets of different size.

### How it works in Python

In [3]:
import numpy as np
import pandas as pd
from scipy import stats

listings = pd.read_parquet("case-study/data/airbnb/listings.parquet")
short = listings[listings["price"].notna() & listings["minimum_nights"].lt(28)]

def compare(df: pd.DataFrame, column: str, group: str, a_value, b_value) -> dict:
    """Welch t-test, Mann-Whitney U and effect sizes for two groups of one column."""
    a = df.loc[df[group] == a_value, column].dropna()
    b = df.loc[df[group] == b_value, column].dropna()
    t = stats.ttest_ind(a, b, equal_var=False)
    ci = t.confidence_interval()
    mw = stats.mannwhitneyu(a, b)
    pooled_sd = np.sqrt(((len(a) - 1) * a.var() + (len(b) - 1) * b.var()) / (len(a) + len(b) - 2))
    return {"diff": round(float(a.mean() - b.mean()), 2),
            "ci": (round(float(ci.low), 2), round(float(ci.high), 2)),
            "p_t": float(f"{t.pvalue:.2g}"), "p_mw": float(f"{mw.pvalue:.2g}"),
            "d": round(float((a.mean() - b.mean()) / pooled_sd), 2),
            "cles": round(float(mw.statistic / (len(a) * len(b))), 3)}

print(compare(short, "price", "room_type", "Entire home/apt", "Private room"))
# {'diff': 110.33, 'ci': (102.4, 118.26), 'p_t': 5.6e-155, 'p_mw': 0.0, 'd': 0.52, 'cles': 0.836}
print(compare(short, "price", "district", "Mitte", "Neukölln"))
# {'diff': 65.47, 'ci': (54.48, 76.46), 'p_t': 2.6e-30, 'p_mw': 7e-33, 'd': 0.47, 'cles': 0.671}
print(compare(short, "price", "host_is_superhost", True, False))
# {'diff': 7.88, 'ci': (-1.85, 17.62), 'p_t': 0.11, 'p_mw': 0.00012, 'd': 0.04, 'cles': 0.527}
print(compare(listings, "review_scores_rating", "host_is_superhost", True, False))
# {'diff': 0.14, 'ci': (0.13, 0.15), 'p_t': 3.7e-133, 'p_mw': 1.7e-40, 'd': 0.41, 'cles': 0.581}

{'diff': 110.33, 'ci': (102.4, 118.26), 'p_t': 5.6e-155, 'p_mw': 0.0, 'd': 0.52, 'cles': 0.836}
{'diff': 65.47, 'ci': (54.48, 76.46), 'p_t': 2.6e-30, 'p_mw': 7e-33, 'd': 0.47, 'cles': 0.671}
{'diff': 7.88, 'ci': (-1.85, 17.62), 'p_t': 0.11, 'p_mw': 0.00012, 'd': 0.04, 'cles': 0.527}
{'diff': 0.14, 'ci': (0.13, 0.15), 'p_t': 3.7e-133, 'p_mw': 1.7e-40, 'd': 0.41, 'cles': 0.581}


Reading the four results:

- **Entire home vs private room**: entire homes cost €110 more per night on average (95 % CI €102 to €118). Both tests agree, and the CLES of 0.84 says the difference holds for most pairs of listings. The room type is the first thing any price comparison must take into account.
- **Mitte vs Neukölln**: €65 more in Mitte (CI €54 to €76), CLES 0.67: a clear but overlapping difference. The test cannot say *why*. Mitte has more entire homes and larger listings (page 4 asks how much of the gap remains when room type and size are held fixed).
- **Superhost vs other hosts, price**: the two tests disagree. The t-test finds no clear difference in means (p = 0.11, CI −€2 to €18), the Mann–Whitney test does (p = 0.0001). Both are right about their own question: superhost listings tend to be slightly more expensive (CLES 0.53), but the means are dominated by a few very expensive listings, which makes the t-test noisy. Either way the effect is negligible.
- **Superhost vs other hosts, rating**: superhosts are rated 0.14 stars higher (CI 0.13 to 0.15). Tiny p-values, a moderate d of 0.41, a CLES of 0.58. Whether 0.14 stars matters depends on how crowded the ratings are: with a median of 4.86, it is a noticeable step. Superhost status is itself awarded partly for high ratings, so this comparison is close to circular.

### In practice

- Clinical trials report the treatment effect in clinical units (for example mmHg of blood pressure) with a 95 % CI, as required by the CONSORT statement.
- Usability studies compare task-completion times of two interface designs; times are skewed, so the Mann–Whitney test or a log transformation is common.
- Paired t-tests are standard in before–after evaluations of training programmes.

> **Warning.** **Counts and ordinal scales.** The review score is an average of star ratings, bounded at 5 and crowded near the top; a t-test treats it as interval data. This is common and usually harmless with large samples, but the Mann–Whitney test, or the share of listings rated below 4.5, is easier to defend. The same holds for small counts such as the number of guests. Report what the reader cares about.

## Categorical data: contingency tables, chi-square and Cramér's V

### Concept

A **contingency table** counts observations for every combination of two categorical variables. The **chi-square test of independence** compares the observed counts O with the counts E expected if the variables were unrelated:

E = row total × column total / grand total, and χ² = Σ (O − E)² / E.

Large gaps give a large χ² and a small p-value. **Cramér's V** = √(χ² / (n · (k − 1))), with k the smaller number of rows or columns, rescales χ² to a strength between 0 (no association) and 1 (perfect association). Rough guide for tables with k = 2: 0.1 small, 0.3 medium, 0.5 large.

Worked example (hosts with several listings × an entry in the licence field, all 12,776 listings): 4,737 listings of multi-listing hosts have an entry. Expected under independence: 5,886 listings of multi-listing hosts × 8,809 listings with an entry / 12,776 = 4,058. The observed count is 679 above expectation: about 17 %.

If any expected count is below about 5, use **Fisher's exact test** (2 × 2 tables) instead.

### Why it matters

Most business, health and social data are categorical: plan, country, device, diagnosis, yes/no. A 2 × 2 table (variant × converted) is exactly the shape of an A/B test on a conversion rate.

### How it works in Python

In [4]:
import pandas as pd
from scipy import stats
from scipy.stats.contingency import association

listings = pd.read_parquet("case-study/data/airbnb/listings.parquet")
multi = listings["calculated_host_listings_count"].gt(1).rename("multi_listing_host")
entry = listings["license"].notna().rename("licence_entry")        # any entry in the licence field
table = pd.crosstab(multi, entry)
print(table)
# licence_entry       False  True
# multi_listing_host
# False                2818   4072
# True                 1149   4737

res = stats.chi2_contingency(table)
print(round(res.statistic, 1), f"{res.pvalue:.1e}", res.dof)   # 676.7 3.5e-149 1
print(pd.DataFrame(res.expected_freq, index=table.index, columns=table.columns).round(0))
print(round(association(table, method="cramer"), 3))          # 0.23: small to moderate
print(pd.crosstab(multi, entry, normalize="index").round(3))
# licence_entry       False   True
# multi_listing_host
# False               0.409  0.591
# True                0.195  0.805   -> 81 % against 59 %

# a larger table: 12 districts x 4 room types
big = pd.crosstab(listings["district"], listings["room_type"])
res = stats.chi2_contingency(big)
print(big.shape, res.dof, f"{res.pvalue:.1e}", round(association(big, method="cramer"), 3))   # (12, 4) 33 3.4e-42 0.086
print((res.expected_freq < 5).sum())                          # 12 cells expected below 5

licence_entry       False  True 
multi_listing_host              
False                2818   4072
True                 1149   4737
676.7 3.5e-149 1
licence_entry        False   True 
multi_listing_host                
False               2139.0  4751.0
True                1828.0  4058.0
0.23
licence_entry       False  True 
multi_listing_host              
False               0.409  0.591
True                0.195  0.805
(12, 4) 33 3.4e-42 0.086
12


**Two significant results of different weight.** Listings of hosts with several listings carry an entry in the licence field far more often (81 % against 59 %); V = 0.23 is a small-to-moderate association with a clear practical meaning: registration details are more complete among professional hosts. Berlin has required a registration number for holiday rentals for years, and an EU regulation on short-term rental data (Regulation (EU) 2024/1028) applies from 20 May 2026; the data cannot tell why private hosts leave the field empty more often. Note also what the field contains: for many multi-listing hosts it is the name of a legal entity rather than a registration number (`license_status`; Session 4, workbook 07), so "licence entry" is not the same as "registered". The district × room type table is also highly significant (p = 3.4e-42), but V = 0.086 is small: the room-type mix differs between districts (more private rooms in Neukölln and Reinickendorf), but not dramatically. Twelve of 48 cells, mostly hotel and shared rooms, have expected counts below 5, so that p-value is approximate; V is still a useful description.

### In practice

- Public-health reporting cross-tabulates vaccination status and hospitalisation.
- Churn analysis compares cancellation rates by subscription plan (Session 6 uses the IBM Telco data).
- Housing authorities cross-tabulate short-term rental listings by district and type to monitor where whole flats are withdrawn from the rental market.
- Recruitment audits compare offer rates by applicant group with contingency tables; the Berkeley admissions case in [03-lecture-relationships.ipynb](03-lecture-relationships.ipynb) shows why such tables need a closer look.

> **Tip.** Always print the **row or column shares** next to the test. The test tells you whether the table differs from independence; the shares tell you how.

> 📖 **Read more:** [extras/13-categorical-relationships.ipynb](extras/13-categorical-relationships.ipynb) (statsthinking21): computes the chi-square statistic by hand from the expected counts of a smoking × cancer table (NHANES) and reads the standardised residuals to see which cells drive the result.

## A/B tests as an application

### Concept

An **A/B test** is a randomised controlled experiment. Users are randomly assigned to the current version (A, control) or the change (B, treatment). Example used below: a booking platform tests a new photo gallery on its listing pages; the metric is the share of visitors who send a booking request. Randomisation makes the groups alike in everything else, so a difference in outcome is caused by the change.

Before launch, write down:

- **one primary metric** (for example the conversion rate) and **guardrail metrics** that must not get worse (load time, cancellations);
- the **minimum detectable effect (MDE)**: the smallest lift worth acting on;
- **α** (accepted false-positive rate, usually 5 %) and **power** (the chance to detect the MDE if it is real, usually 80 %).

Together these determine the **sample size**. After the test, analyse the 2 × 2 table (variant × converted) and give a CI for the lift.

```mermaid
sequenceDiagram
    participant P as Product team
    participant E as Experiment platform
    participant U as Users
    P->>E: metric, MDE, alpha, power -> sample size
    E->>U: random assignment to A or B
    U-->>E: conversions per group
    E->>E: sample-ratio check, then test
    E-->>P: lift with 95 % CI and guardrails
```

### Why it matters

Experiments are where data roles use statistics most often. Product teams in e-commerce and software run tests continuously, and interviews for analyst roles routinely include an A/B-testing case. Two common errors invalidate results: **peeking** (checking every day and stopping at the first p < 0.05, which raises the false-positive rate far above 5 %) and a **sample-ratio mismatch** (the observed split differs from the planned one, a sign of broken assignment or logging).

### How it works in Python

In [5]:
from scipy import stats
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import (confint_proportions_2indep, proportion_effectsize,
                                          proportions_ztest)

# Plan: baseline booking-request rate 10 %, smallest lift worth acting on: 10 % -> 11 %
effect = proportion_effectsize(0.11, 0.10)
n = NormalIndPower().solve_power(effect_size=effect, alpha=0.05, power=0.8)
print(round(n))                                            # 14744 users per group

# Analyse after the planned sample is reached
conversions, users = [1_630, 1_480], [14_800, 14_750]      # B (new gallery), A (current page); simulated
z, p = proportions_ztest(conversions, users)
print(round(z, 2), round(p, 4))                            # 2.74 0.0061
low, high = confint_proportions_2indep(conversions[0], users[0], conversions[1], users[1])
print(round(low, 4), round(high, 4))                       # 0.0028 0.0168: +0.3 to +1.7 points

# Sample-ratio check: a planned 50/50 split
print(stats.chisquare(users).pvalue.round(3))              # 0.771: no mismatch

14744
2.74 0.0061
0.0028 0.0168
0.771


### In practice

- Kohavi, Tang and Xu (2020) describe experimentation at Microsoft (Bing), Google and LinkedIn, where thousands of controlled experiments run each year.
- The UK Behavioural Insights Team ran randomised trials of tax reminder letters with HM Revenue and Customs and found that social-norm messages ("most people pay on time") raised payment rates.
- Booking.com has described running about 1,000 concurrent experiments on its website (Thomke, 2020); Airbnb's data science team described how stopping its search-page experiments at the first significant result would have produced false wins (Overgoor, 2014).

> **Warning.** **Peeking.** If you must look at results before the planned sample size, use a method designed for repeated looks (sequential testing). Otherwise fix the sample size in advance and analyse once.

## Choosing a test from a decision table

### Concept

The choice depends on three things: the type of the **outcome**, the number of **groups**, and whether the groups are **independent** or **paired** (the same units measured twice).

| Outcome | Groups | Independent groups | Paired / repeated |
|---|---|---|---|
| numeric, roughly symmetric or large n | 2 | Welch t-test (`ttest_ind(equal_var=False)`) | paired t-test (`ttest_rel`) |
| numeric, skewed, small n, or ordinal | 2 | Mann–Whitney U (`mannwhitneyu`) | Wilcoxon signed-rank (`wilcoxon`) |
| numeric | 3 or more | one-way ANOVA (`f_oneway`) or Welch ANOVA | repeated-measures ANOVA |
| ordinal or skewed | 3 or more | Kruskal–Wallis (`kruskal`) | Friedman (`friedmanchisquare`) |
| categorical | 2 or more | chi-square (`chi2_contingency`); Fisher's exact for small 2 × 2 | McNemar (2 × 2, `statsmodels`) |
| proportion (A/B) | 2 | two-proportion z-test (`proportions_ztest`) | McNemar |
| two numeric variables | – | Pearson r (linear) or Spearman ρ (monotonic), [03-lecture-relationships.ipynb](03-lecture-relationships.ipynb) | – |

The same choice as a decision flowchart:

```mermaid
flowchart TD
    S{"Outcome type?"} -->|"numeric or ordinal"| G{"How many groups?"}
    S -->|"categorical"| P1{"Paired?"}
    S -->|"two numeric variables"| COR["Pearson r (linear)<br/>Spearman rho (monotonic)"]
    P1 -->|"no"| SMALL{"Any expected<br/>count below 5?"}
    P1 -->|"yes, 2 x 2"| MCN["McNemar"]
    SMALL -->|"no"| CHI["chi-square test<br/>+ Cramer's V"]
    SMALL -->|"yes, 2 x 2"| FISH["Fisher's exact test"]
    G -->|"two"| P2{"Paired?"}
    G -->|"three or more"| P3{"Roughly normal<br/>or large n?"}
    P2 -->|"no"| N2{"Roughly normal<br/>or large n?"}
    P2 -->|"yes"| N3{"Roughly normal<br/>differences?"}
    N2 -->|"yes"| WT["Welch t-test<br/>+ CI, Cohen's d"]
    N2 -->|"no, or ordinal"| MW["Mann-Whitney U<br/>+ CLES"]
    N3 -->|"yes"| PT["paired t-test"]
    N3 -->|"no"| WS["Wilcoxon signed-rank"]
    P3 -->|"yes"| AN["one-way ANOVA<br/>then post-hoc tests"]
    P3 -->|"no, or ordinal"| KW["Kruskal-Wallis"]
```

### Why it matters

A wrong test can give a wrong answer: an independent-samples test on paired data ignores that each person is compared with themselves and loses power; a chi-square test on tiny counts gives unreliable p-values. A decision table makes the choice explicit and reviewable.

### How it works in Python

In [6]:
import pandas as pd
from scipy import stats

listings = pd.read_parquet("case-study/data/airbnb/listings.parquet")
short = listings[listings["price"].notna() & listings["minimum_nights"].lt(28)]

# Three or more groups, skewed outcome -> Kruskal-Wallis: does the price differ by district?
groups = [g["price"] for _, g in short.groupby("district")]
print(stats.kruskal(*groups).pvalue < 0.001)                         # True
print(short.groupby("district")["price"].median().sort_values().iloc[[0, 1, -2, -1]].to_dict())
# {'Reinickendorf': 99.535, 'Treptow - Köpenick': 125.0, 'Pankow': 174.0, 'Mitte': 187.0}

# Small 2 x 2 table -> Fisher's exact test
print(round(stats.fisher_exact([[8, 2], [1, 5]]).pvalue, 3))        # 0.035

True
{'Reinickendorf': 99.535, 'Treptow - Köpenick': 125.0, 'Pankow': 174.0, 'Mitte': 187.0}
0.035


### In practice

- Statistical analysis plans for clinical trials name the test for every endpoint before data collection.
- Journals such as *Nature* require authors to state the test used and whether it was one- or two-sided for every reported p-value.
- Experimentation platforms fix the test per metric type (proportion, mean, ratio) so that analysts do not choose after seeing the data.

> **Important.** **Worked example.** Part B of [04-case-study-airbnb-exploration.ipynb](04-case-study-airbnb-exploration.ipynb) compares entire homes with private rooms and Mitte with Neukölln (Welch's t-test, Mann–Whitney test, CI, Cohen's d, CLES), tests multi-listing host × licence entry and district × room type with chi-square and Cramér's V, and simulates the photo-gallery A/B test with a sample-ratio check.

> **Caution.** **Many tests.** At α = 0.05, each test on pure noise has a 5 % chance of a false positive; with 20 tests the chance of at least one is 64 %. When you test many metrics or subgroups, correct with Holm or Benjamini–Hochberg (`statsmodels.stats.multitest.multipletests`).

> 📖 **Read more:** [extras/12-one-way-anova.ipynb](extras/12-one-way-anova.ipynb) (statsintro_python, Haslwanter): the test for three or more groups from the decision table: why the spread within groups matters, a Levene test for equal variances, and a one-way ANOVA with scipy and with statsmodels (`anova_lm`).

## From correlation to the regression line

### Concept

The **least-squares line** ŷ = b₀ + b₁·x is the straight line that minimises the sum of squared vertical distances (**residuals** y − ŷ) between the points and the line. Its slope and intercept follow directly from the correlation:

b₁ = r · s_y / s_x and b₀ = ȳ − b₁ · x̄,

where s_x and s_y are the standard deviations. The line always passes through the point of means (x̄, ȳ). In simple regression, the **R²** of the line equals r².

![Left: a least-squares line with residuals on toy data. Right: log price against the number of guests for short-stay listings, with the mean per number of guests and the least-squares line](../figures/regression-line.png)

Worked example: if r = 0.5, s_x = 2 and s_y = 10, then b₁ = 0.5 × 10 / 2 = 2.5: y rises by 2.5 units per unit of x. The correlation is symmetric in x and y; the slope is not.

### Why it matters

The regression line turns "these variables move together" into "how much y changes per unit of x", in the units of the data, and gives a prediction for new x values. Session 6 builds on it: several predictors, training and test data, error metrics.

### How it works in Python

In [7]:
import numpy as np
import pandas as pd

listings = pd.read_parquet("case-study/data/airbnb/listings.parquet")
short = listings[listings["price"].notna() & listings["minimum_nights"].lt(28)]
x = short["accommodates"]
y = np.log(short["price"])           # natural log: coefficients read as approximate percentages

r = np.corrcoef(x, y)[0, 1]
b1 = r * y.std() / x.std()            # slope from the correlation
b0 = y.mean() - b1 * x.mean()         # line through the point of means
print(round(r, 3), round(b1, 3), round(b0, 3))       # 0.594 0.151 4.526
print(np.polyfit(x, y, deg=1).round(3))              # [0.151 4.526]: numpy agrees
residuals = y - (b0 + b1 * x)
r2 = 1 - (residuals**2).sum() / ((y - y.mean())**2).sum()
print(round(r2, 3), round(r**2, 3))                  # 0.353 0.353: R² = r² in simple regression
print(round(np.exp(b1) - 1, 3), round(np.exp(b0 + b1 * 2), 0), round(np.exp(b0 + b1 * 4), 0))   # 0.163 125.0 169.0

0.594 0.151 4.526
[0.151 4.526]
0.353 0.353
0.163 125.0 169.0


With y on the natural-log scale, a slope of 0.151 means: each additional guest goes with a price about e^0.151 − 1 ≈ 16 % higher. The line predicts €125 for a listing for two guests and €169 for four (these are typical, geometric-mean prices, not means). R² = 0.35: the number of guests explains about a third of the variation in log price; room type, location and much else explain the rest (Session 6). The binned means in the figure bend below the line for large listings: the relationship flattens above about eight guests.

### In practice

- Galton (1886) introduced the term "regression" when he fitted a line to the heights of parents and their adult children and found that children of tall parents were, on average, less extreme ("regression towards the mean").
- Laboratory calibration curves map an instrument signal to a known concentration with a least-squares line.
- Hedonic price indices at statistical offices start from regressions of price on product characteristics.

> 📖 **Read more:** [extras/10-correlation-and-simple-regression.ipynb](extras/10-correlation-and-simple-regression.ipynb) (statsintro_python, Haslwanter) and [extras/14-seaborn-regression.ipynb](extras/14-seaborn-regression.ipynb) (seaborn documentation) show the same bridge in scipy, statsmodels and seaborn (`regplot`, `lmplot`, residual plots).

## Communicating findings: a short report or a Streamlit dashboard

### Concept

A finding for a non-technical reader follows a fixed order:

1. **the finding first**, as a full sentence ("A night in Mitte costs about a fifth more than a comparable night in Neukölln");
2. **one number with its context** (median short-stay price €187 in Mitte and €130 in Neukölln; holding room type and number of guests fixed, Neukölln is about 20 % cheaper);
3. **what it means for the reader's decision** (for a city housing analyst: the raw gap overstates the location premium, because Neukölln offers more private rooms);
4. **one limitation** (listed prices, not paid prices; one snapshot from June 2026; medium-term listings excluded because their price field is not comparable).

A **one-page report** puts three to five such findings on a page, each with one chart. A **dashboard** gives repeated access to the same analysis with the reader's own filters.

**Streamlit** turns a Python script into a web app. The script runs from top to bottom; widgets such as `st.slider` return the current user input, and `st.metric` or `st.plotly_chart` display results. Every time the user changes a widget, the whole script runs again. Expensive steps are therefore **cached** with `@st.cache_data`. The app is started with `streamlit run app.py` and opens at `http://localhost:8501`.

```mermaid
sequenceDiagram
    participant U as Housing analyst
    participant B as Browser
    participant S as Streamlit script
    U->>B: picks districts and a room type
    B->>S: rerun with new widget values
    S->>S: load data (cached), filter
    S-->>B: metrics and charts
    B-->>U: updated page
```

### Why it matters

Most analyses reach decision makers as one chart and a few sentences. A finding that is correct but not understood has no effect. A dashboard avoids repeated one-off requests ("can you rerun this for private rooms in Pankow?") and is one option for the user interface of a deployed model in Session 16.

### How it works in Python

The complete app is [`workbooks/dashboard_app.py`](dashboard_app.py). Its core:

In [8]:
# dashboard core: start the full app with  streamlit run sessions/05-eda-and-statistics/workbooks/dashboard_app.py
import pandas as pd
import plotly.express as px
import streamlit as st

@st.cache_data                     # read the file once, not on every interaction
def load() -> pd.DataFrame:
    listings = pd.read_parquet("case-study/data/airbnb/listings.parquet")
    return listings[listings["price"].notna() & listings["minimum_nights"].lt(28)]

short = load()
st.title("Short-stay listings in Berlin: prices by district")
room_types = st.sidebar.multiselect("Room types", sorted(short["room_type"].unique()), default=["Entire home/apt"])
max_guests = st.sidebar.slider("Guests up to", 1, 16, 4)

sel = short[short["room_type"].isin(room_types) & short["accommodates"].le(max_guests)]
st.metric("Listings", f"{len(sel):,}")
by_district = sel.groupby("district", as_index=False)["price"].median().sort_values("price")
st.plotly_chart(px.bar(by_district, x="price", y="district", orientation="h"))

2026-10-11 23:08:36.241 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-10-11 23:08:36.242 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-10-11 23:08:36.243 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-10-11 23:08:36.254 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-10-11 23:08:36.370 
  command:

    streamlit run .venv/lib/python3.14/site-packages/ipykernel_launcher.py [ARGUMENTS]


...

2026-10-11 23:08:36.444 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-10-11 23:08:36.445 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-10-11 23:08:36.445 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-10-11 23:08:36.446 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

Run outside Streamlit, the script prints warnings and draws nothing; that is expected. Start it with `streamlit run`.

### In practice

- Snowflake acquired Streamlit in 2022 and offers it inside its data platform for internal data apps.
- Hugging Face Spaces hosts thousands of public Streamlit and Gradio demos of machine-learning models.
- The UK Office for National Statistics and Eurostat publish short statistical bulletins that follow the "main points first" structure.

> **Important.** **Worked example.** Part C of [04-case-study-airbnb-exploration.ipynb](04-case-study-airbnb-exploration.ipynb) measures how strongly the price rises with the number of guests (Pearson, Spearman, log scale) and checks whether district price differences survive once room type, size and the type of stay are taken into account. The exercises of this block ask for a one-page finding for a city housing analyst or an extension of the dashboard.

> **Warning.** **Dashboards spread mistakes.** A wrong filter or an unlabelled axis in a dashboard is seen by everyone who opens it, every day. Test the numbers against a notebook, label units, and show the data period and the number of observations on the page.

## Check your understanding

1. A test gives p = 0.03. Which of these statements are correct: "H₀ is false with probability 97 %", "If H₀ were true, data this extreme would occur about 3 % of the time"?
2. For the superhost price comparison, the t-test gives p = 0.11 and the Mann–Whitney test p = 0.0001. How can both be right, and what would you report?
3. Compute the expected number of listings of single-listing hosts **without** a licence entry under independence from the table above, and compare it with the observed 2,818.
4. If r = −0.4, s_x = 5 and s_y = 2, what is the slope of the least-squares line?
5. Write the four-part summary (finding, number, meaning, limitation) for the comparison of entire homes and private rooms.

## Further reading

- Downey, A. B. (2025). *Think Stats* (3rd ed.), chapter 9 "Hypothesis testing". <https://allendowney.github.io/ThinkStats/chap09.html>
- Wasserstein, R. L., & Lazar, N. A. (2016). The ASA statement on p-values: Context, process, and purpose. *The American Statistician*, 70(2), 129–133. <https://doi.org/10.1080/00031305.2016.1154108>
- Kohavi, R., Tang, D., & Xu, Y. (2020). *Trustworthy Online Controlled Experiments: A Practical Guide to A/B Testing*. Cambridge University Press. <https://experimentguide.com/>
- Miller, E. (2010). *How not to run an A/B test*. <https://www.evanmiller.org/how-not-to-run-an-ab-test.html>
- Streamlit (2026). *Get started: tutorials*. <https://docs.streamlit.io/get-started/tutorials>